# Latihan Assignment Speech Recognition
Implementasi ringkas dan efisien untuk menjawab permintaan dokumen:
1. **Audio Augmentation**: Pitch shift, White noise, Repeat clip, dan Fade in dengan validasi input dan ekspor ke `.wav`.
2. **Predict Text**: Pengenalan suara menggunakan model **Wav2Vec2** dan penggantian delimiter `'|'` menjadi spasi.


### 1. Inisialisasi Audio & Model Wav2Vec2
Membaca file audio dasar dan memuat model Wav2Vec2 dari `torchaudio`.


In [1]:
import warnings, numpy as np, soundfile as sf, librosa, torch, torchaudio
warnings.filterwarnings('ignore')

AUDIO_PATH = 'sample.wav'
sig, sr = sf.read(AUDIO_PATH)
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()


### 2. Fungsi Validasi Input & Prediksi Teks
- `get_num()`: Validasi angka dalam range tertentu menggunakan *lambda*.
- `get_wav()`: Validasi nama file berakhiran `.wav`.
- `predict_text()`: Decoding emisi Wav2Vec2 langsung ke teks tanpa class tambahan.


In [2]:
def get_num(prompt, cond, t=float):
    while True:
        try:
            if cond(v := t(input(prompt).strip())): return v
        except ValueError: pass
        print("Input tidak valid!")

def get_wav():
    while not (f := input("Input new file name: ").strip()).endswith(".wav"):
        print("Nama file harus berakhiran '.wav'!")
    return f

def predict_text():
    wf, s_rate = torchaudio.load(AUDIO_PATH)
    if s_rate != bundle.sample_rate:
        wf = torchaudio.functional.resample(wf, s_rate, bundle.sample_rate)
    with torch.inference_mode():
        emission, _ = model(wf.mean(dim=0, keepdim=True))
    chars = [bundle.get_labels()[i] for i in torch.unique_consecutive(torch.argmax(emission[0], dim=-1)) if i != 0]
    print("Predicted text:", "".join(chars).replace("|", " "))


### 3. Sub-Menu Audio Augmentation
Menjalankan 4 jenis augmentasi audio dan mengekspor hasilnya ke file `.wav` baru.


In [3]:
def audio_augmentation():
    while True:
        print("\n1. Pitch shift (Increase or decrease pitch)\n2. Add white noise\n3. Repeat clip\n4. Insert fade in\n5. Back to main menu")
        sub = input("Input your choice: ").strip()
        if sub == '1':
            rate = get_num("Input pitch rate [-5 to 5]: ", lambda x: -5 <= x <= 5)
            sf.write(get_wav(), librosa.effects.pitch_shift(sig, sr=sr, n_steps=rate), sr)
        elif sub == '2':
            lvl = get_num("Input noise level [0.1 to 0.5]: ", lambda x: 0.1 <= x <= 0.5)
            sf.write(get_wav(), sig + np.random.normal(0, sig.std(), sig.shape) * lvl, sr)
        elif sub == '3':
            rep = get_num("Input repeat count (> 0): ", lambda x: x > 0, int)
            sf.write(get_wav(), np.tile(sig, rep), sr)
        elif sub == '4':
            dur = get_num("Input duration effect (> 0): ", lambda x: x > 0)
            fade_len = min(int((dur / 1000 if dur > 50 else dur) * sr), len(sig))
            res = sig.copy(); res[:fade_len] *= np.linspace(0, 1, fade_len)
            sf.write(get_wav(), res, sr)
        elif sub == '5':
            break


### 4. Menu Utama Aplikasi
Loop interaktif menu utama: 1. Audio Augmentation, 2. Predict Text, 3. Exit.


In [4]:
while True:
    print("\n1. Audio Augmentation\n2. Predict Text\n3. Exit")
    c = input("Input your choice: ").strip()
    if c == '1': audio_augmentation()
    elif c == '2': predict_text()
    elif c == '3': break



1. Audio Augmentation
2. Predict Text
3. Exit
Predicted text: I REALLY LOVE TITE SO I LIKE TO HUG TETE 

1. Audio Augmentation
2. Predict Text
3. Exit
